In [9]:
import pandas as pd
p = pd.read_csv("../data/penguins_raw.csv")
p[p["Comments"] == "Not enough blood for isotopes."][["Sex", "Delta 15 N (o/oo)", "Delta 13 C (o/oo)"]]

,Sex,Delta 15 N (o/oo),Delta 13 C (o/oo)
0,MALE,NaN,NaN
12,FEMALE,NaN,NaN
13,MALE,NaN,NaN
15,FEMALE,NaN,NaN
41,MALE,NaN,NaN
46,MALE,NaN,NaN
182,MALE,NaN,NaN


In [10]:
p[p["Comments"] == "No blood sample obtained for sexing."][["Sex", "Delta 15 N (o/oo)", "Delta 13 C (o/oo)"]]

,Sex,Delta 15 N (o/oo),Delta 13 C (o/oo)
9,NaN,9.13362,-25.09368
10,NaN,8.63243,-25.21315


In [11]:
orders = pd.read_csv("../data/orders_joined.csv")
pd.DataFrame({"type": orders.dtypes.astype(str),
 "distinct": orders.nunique(),
 "missing": orders.isna().sum()})

,type,distinct,missing
order_id,str,1996,0
order_date,str,362,0
customer_id,int64,907,0
channel,str,2,0
category,str,6,0
item_price,float64,1829,0
quantity,int64,5,0
discount_pct,int64,8,0
order_value,float64,1872,0
payment_method,str,4,0


In [12]:
penguins = pd.read_csv("../data/penguins_raw.csv")
pd.DataFrame({"type": penguins.dtypes.astype(str),
 "distinct": penguins.nunique(),
 "missing": penguins.isna().sum()})

,type,distinct,missing
studyName,str,3,0
Sample Number,int64,152,0
Species,str,3,0
Region,str,1,0
Island,str,3,0
Stage,str,1,0
Individual ID,str,190,0
Clutch Completion,str,2,0
Date Egg,str,50,0
Culmen Length (mm),float64,164,2


In [14]:
print(p.duplicated(["studyName", "Individual ID"]).sum())
print(p.duplicated(["studyName", "Sample Number"]).sum())
print(p.duplicated(["studyName", "Species", "Sample Number"]).sum())

0
124
0


In [16]:
import pandas as pd

full = pd.read_csv("../data/orders_joined.csv")
exp = full.item_price * full.quantity * (1 - full.discount_pct / 100)

exact = (full.order_value != exp).sum()
penny = ((full.order_value - exp).abs() >= 0.01).sum()

print("flagged by ==            ", exact)
print("flagged within a penny   ", penny)
print("rounding only            ", exact - penny)

flagged by ==             741
flagged within a penny    12
rounding only             729


In [19]:
import pandas as pd

full = pd.read_csv("../data/orders_joined.csv")
exp = full.item_price * full.quantity * (1 - full.discount_pct / 100)
checks = pd.DataFrame({
    "quantity": full.quantity < 1,
    "price": full.item_price <= 0,
    "delivery": full.delivery_days.notna() & ~full.delivery_days.between(0, 60),
    "value": full.order_value < 0,
    "identity": (full.order_value - exp).abs() >= 0.01,
})
bad = checks[checks.any(axis=1)]
print(bad.sum())
print("delivery is exactly -999:", (full.delivery_days == -999).sum())
print(pd.crosstab(bad.drop(columns="identity").any(axis=1), bad["identity"],
                  rownames=["fails a range rule"], colnames=["fails identity"]))
full.loc[bad.index, ["order_id", "item_price", "quantity", "discount_pct",
                     "order_value", "delivery_days"]].join(bad.add_prefix("fails_"))

quantity     4
price        4
delivery     5
value        4
identity    12
dtype: int64
delivery is exactly -999: 5
fails identity      False  True 
fails a range rule              
True                    5     12


,order_id,item_price,quantity,discount_pct,order_value,delivery_days,fails_quantity,fails_price,fails_delivery,fails_value,fails_identity
178,ORD-100182,353.98,2,10,637.16,-999,False,False,True,False,False
229,ORD-100234,43.71,1,20,-34.97,5,False,False,False,True,True
243,ORD-100250,14.63,1,0,14.63,-999,False,False,True,False,False
309,ORD-100320,0.00,2,0,79.20,5,False,True,False,False,True
364,ORD-100378,69.17,2,0,-138.34,3,False,False,False,True,True
420,ORD-100438,0.00,2,0,18.56,6,False,True,False,False,True
425,ORD-100443,93.46,0,0,186.92,4,True,False,False,False,True
613,ORD-100647,0.00,1,0,62.26,2,False,True,False,False,True
691,ORD-100729,56.62,1,10,50.96,-999,False,False,True,False,False
1069,ORD-101124,133.38,1,0,133.38,-999,False,False,True,False,False


In [18]:
p = pd.read_csv("../data/penguins_raw.csv")
n_species = p.groupby("Individual ID")["Species"].nunique()
two = n_species[n_species > 1].index
print("IDs with two species:", len(two), " rows carrying them:", p["Individual ID"].isin(two).sum())

IDs with two species: 94  rows carrying them: 228


In [20]:
full.loc[full.order_value < 0, ["order_id", "returned"]]

,order_id,returned
229,ORD-100234,1
364,ORD-100378,0
1519,ORD-101594,0
1782,ORD-101872,0


In [21]:
cols = ["Culmen Length (mm)", "Culmen Depth (mm)", "Flipper Length (mm)",
        "Body Mass (g)", "Sex", "Delta 15 N (o/oo)", "Delta 13 C (o/oo)"]
inc = p[p[cols].isna().any(axis=1)]
print("incomplete rows:", len(inc))
print(inc["studyName"].value_counts())
inc.groupby(inc["Comments"].fillna("(no comment)"))[cols].apply(lambda g: g.isna().sum())

incomplete rows: 20
studyName
PAL0708    15
PAL0910     4
PAL0809     1
Name: count, dtype: int64


,Culmen Length (mm),Culmen Depth (mm),Flipper Length (mm),Body Mass (g),Sex,Delta 15 N (o/oo),Delta 13 C (o/oo)
Comments,,,,,,,
Adult not sampled.,1,1,1,1,1,1,1
Adult not sampled. Nest never observed with full clutch.,1,1,1,1,1,1,1
Nest never observed with full clutch. Not enough blood for isotopes.,0,0,0,0,0,1,1
No blood sample obtained for sexing.,0,0,0,0,2,0,0
No blood sample obtained.,0,0,0,0,2,2,2
No delta15N data received from lab.,0,0,0,0,0,1,0
Not enough blood for isotopes.,0,0,0,0,0,7,7
Sexing primers did not amplify.,0,0,0,0,4,0,0
Sexing primers did not amplify. Not enough blood for isotopes.,0,0,0,0,1,1,1


In [22]:
blank = full[full.marketing_opt_in.isna()]
print("orders:", len(blank), " customers:", blank.customer_id.nunique())
print(blank.channel.value_counts())
print("return rate, blank opt-in:", round(blank.returned.mean(), 3),
      " everyone else:", round(full[full.marketing_opt_in.notna()].returned.mean(), 3))

orders: 77  customers: 48
channel
web    77
Name: count, dtype: int64
return rate, blank opt-in: 0.182  everyone else: 0.192


In [23]:
print(len(full))
print(full.isna().sum()[full.isna().sum() > 0])

1996
customer_region         34
customer_tenure_days    34
marketing_opt_in        77
dtype: int64


In [24]:
print((full.marketing_opt_in.isna() & full.customer_region.notna()).sum())

43


In [25]:
crm_blank = full[full.marketing_opt_in.isna() & full.customer_region.notna()]
print(len(crm_blank), crm_blank.customer_id.nunique(), crm_blank.channel.value_counts().to_dict(),
      round(crm_blank.returned.mean(), 3))

43 21 {'web': 43} 0.116


In [26]:
w1 = pd.read_csv("../data/retail_orders_week1.csv")
ids = full.loc[full.marketing_opt_in.isna() & full.customer_region.notna(), "customer_id"].unique()
filled = w1[w1.customer_id.isin(ids)]
print("Week 1 rows for these customers:", len(filled))
print(filled.marketing_opt_in.value_counts().to_dict())
per_cust = filled.groupby("customer_id")["marketing_opt_in"].nunique()
print("customers with more than one order:", (filled.customer_id.value_counts() > 1).sum())
print("customers whose orders disagree:", (per_cust > 1).sum())
print("opt-in share where the CRM has a value:",
      full.marketing_opt_in.value_counts(normalize=True).round(3).to_dict())

Week 1 rows for these customers: 43
{True: 33, False: 10}
customers with more than one order: 15
customers whose orders disagree: 0
opt-in share where the CRM has a value: {True: 0.564, False: 0.436}


In [1]:
import pandas as pd

raw = pd.read_csv("../data/orders_joined.csv")
clean = pd.read_csv("../outputs/orders_clean.csv")

ids = raw.loc[raw["order_value"] < 0, "order_id"]
before = raw[raw["order_id"].isin(ids)].set_index("order_id")
after = clean[clean["order_id"].isin(ids)].set_index("order_id")

check = pd.DataFrame({
    "returned": before["returned"],
    "value_before": before["order_value"],
    "value_after": after["order_value"],
    "sign_corrected": after["order_value_sign_corrected"],
    "ambiguous": after["order_value_ambiguous"],
    "identity": (before["item_price"] * before["quantity"]
                 * (1 - before["discount_pct"] / 100)).round(2),
})
check["magnitude_matches"] = (check["value_before"].abs() - check["identity"]).abs() <= 0.01
check

,returned,value_before,value_after,sign_corrected,ambiguous,identity,magnitude_matches
order_id,,,,,,,
ORD-100234,1,-34.97,-34.97,False,True,34.97,True
ORD-100378,0,-138.34,138.34,True,False,138.34,True
ORD-101594,0,-64.04,64.04,True,False,64.04,True
ORD-101872,0,-21.27,21.27,True,False,21.27,True


In [3]:
o = pd.read_csv("../data/orders_joined.csv")
filled = o["marketing_opt_in"].fillna(o["marketing_opt_in"].mode()[0])
print("as recorded:", o["marketing_opt_in"].value_counts().to_dict(), "unknown", o["marketing_opt_in"].isna().sum())
print("filled:     ", filled.value_counts().to_dict())
print(f"opted-in share {o['marketing_opt_in'].mean():.3f} -> {filled.mean():.3f}")
print(o.loc[o["marketing_opt_in"].isna(), "customer_id"].nunique())

as recorded: {True: 1083, False: 836} unknown 77
filled:      {True: 1160, False: 836}
opted-in share 0.564 -> 0.581
48
